In [1]:
#LSTM
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

TensorFlow version: 2.21.0
Libraries imported successfully!


In [2]:
text = """I love deep learning.Deep learning is amazing and
powerful.Machine learning and artifical intelligence and exicting
fields.Recurrent Neural Networks(RNNs)are well-suited
for tasks involving sequential data,like text generation"""

# Character-level tokenization
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([text])

total_chars = len(tokenizer.word_index) + 1

# Convert text into integer IDs
sequences = tokenizer.texts_to_sequences([text])[0]

print("Unique characters:", total_chars - 1)
print("First 20 character IDs:", sequences[:20])

Unique characters: 30
First 20 character IDs: [4, 3, 6, 12, 16, 1, 3, 9, 1, 1, 17, 3, 6, 1, 5, 7, 2, 4, 2, 10]


In [3]:
sequence_length = 10

def create_sequences(seq, length):
    inputs, targets = [], []

    for i in range(len(seq) - length):
        inputs.append(seq[i:i + length])
        targets.append(seq[i + 1:i + length + 1])

    return np.array(inputs), np.array(targets)


X, y = create_sequences(sequences, sequence_length)

# One-hot encode target characters
y = to_categorical(y, num_classes=total_chars)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (216, 10)
Target shape: (216, 10, 31)


In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(sequence_length,)),

    tf.keras.layers.Embedding(
        input_dim=total_chars,
        output_dim=50
    ),

    tf.keras.layers.LSTM(
        units=100,
        return_sequences=True
    ),

    tf.keras.layers.Dense(
        total_chars,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 50)         │         1,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 10, 100)        │        60,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 31)         │         3,131 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,081 (254.22 KB)

 Trainable params: 65,081 (254.22 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X,
    y,
    epochs=20,
    batch_size=32,
    verbose=1
)

print("LSTM training completed successfully!")

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.0954 - loss: 3.4216
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1019 - loss: 3.3665
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0977 - loss: 3.2325
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1139 - loss: 3.1185
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1458 - loss: 3.0497
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1537 - loss: 3.0102 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1361 - loss: 2.9720
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1690 - loss: 2.9375
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1931 - loss: 2.9009 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1852 - loss: 2.8592 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2028 - loss: 2.8031 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2157 - loss: 2.7406 
E

In [6]:
#generate text
def generate_text(model, tokenizer, seed_text, length=50):
    seed_text = seed_text.lower()

    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    if len(sequence) < sequence_length:
        raise ValueError(
            f"Seed text must contain at least {sequence_length} "
            "known characters."
        )

    sequence = sequence[-sequence_length:]
    result = seed_text

    for _ in range(length):
        x_input = np.array([sequence])

        predictions = model.predict(x_input, verbose=0)

        predicted_id = int(
            np.argmax(predictions[0, -1, :])
        )

        predicted_char = tokenizer.index_word.get(
            predicted_id, " "
        )

        result += predicted_char
        sequence = sequence[1:] + [predicted_id]

    return result


seed_text = "Deep learning"

generated_text = generate_text(
    model,
    tokenizer,
    seed_text,
    length=50
)

print("Generated text:")
print(generated_text)

Generated text:
deep learning and aaring aning and aaring aning and aaring anin
